# 📊 CL-SDRG Phase 4: Evaluation, Ablation & Shortcut Audit

**Target:** Google Colab T4 GPU | **Estimated Time:** 10–15 min

This notebook:
1. Loads the trained checkpoint from Phase 3
2. **Classification**: Accuracy, Macro-F1, Precision, Recall
3. **Retrieval**: Recall@{1,5,20}, MRR@20, nDCG@20 via FAISS
4. **Speaker Flip Rate (SFR)** audit: target < 3%
5. **Ablation baselines**: BM25, Zero-Shot Encoder (kNN)
6. Generates publication-ready result tables and plots

**⚠️ Prerequisites:** Run Notebooks 01 and 03 first.

## 1. Setup

In [ ]:
!pip install -q torch transformers pandas scikit-learn faiss-cpu rank_bm25 tqdm matplotlib seaborn

In [ ]:
import os, sys, random, time, logging
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
import faiss

# Logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers.clear()
h = logging.StreamHandler(sys.stdout)
h.setFormatter(logging.Formatter('[%(asctime)s] %(levelname)-8s %(message)s', datefmt='%H:%M:%S'))
logger.addHandler(h)

SEED = 42; random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logging.info(f'Device: {device}')

def fmt(n): return f'{n:,}'

# Config
ENCODER_NAME = 'intfloat/multilingual-e5-base'
EMBEDDING_DIM = 768; MAX_SEQ_LEN = 128
FGA_HIDDEN = 256; CLS_HIDDEN = 256; CLS_DROPOUT = 0.1; NUM_CLASSES = 3
LABEL2ID = {'TRUE': 0, 'FALSE': 1, 'MIXED': 2}
ID2LABEL = {v:k for k,v in LABEL2ID.items()}
K_VALUES = [1, 5, 20]
SFR_TARGET = 0.03; SFR_PERTURBS = 10; BATCH_SIZE = 16
PROC_DIR = Path('outputs/processed_data')
CKPT_DIR = Path('outputs/checkpoints')
FIG_DIR = Path('outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('✅ Setup complete')

## 2. Model Architecture (same as Phase 3)

In [ ]:
class FrozenEncoder(nn.Module):
    def __init__(self, name=ENCODER_NAME):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        self.encoder = AutoModel.from_pretrained(name)
        for p in self.encoder.parameters(): p.requires_grad = False
        self.encoder.eval()
    @torch.no_grad()
    def encode(self, texts, dev):
        prefixed = [f'query: {t}' for t in texts]
        tok = self.tokenizer(prefixed, max_length=MAX_SEQ_LEN, padding=True, truncation=True, return_tensors='pt').to(dev)
        out = self.encoder(**tok)
        m = tok['attention_mask'].unsqueeze(-1).float()
        return (out.last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1e-9)

class FeatureGatingAgent(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding_dim = EMBEDDING_DIM
        self.net = nn.Sequential(nn.Linear(EMBEDDING_DIM*3, FGA_HIDDEN), nn.ReLU(True),
                                nn.Linear(FGA_HIDDEN, EMBEDDING_DIM*3), nn.Sigmoid())
    def forward(self, eq, es, et):
        return self.net(torch.cat([eq,es,et],-1)).split(self.embedding_dim,-1)

class GatedFusion(nn.Module):
    def forward(self, eq,es,et,aq,as_,at): return aq*eq + as_*es + at*et

class VeracityClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.clf = nn.Sequential(nn.LayerNorm(EMBEDDING_DIM), nn.Linear(EMBEDDING_DIM, CLS_HIDDEN),
                                nn.ReLU(True), nn.Dropout(CLS_DROPOUT), nn.Linear(CLS_HIDDEN, NUM_CLASSES))
    def forward(self, x): return self.clf(x)

print('✅ Architecture defined')

## 3. Load Data & Embeddings

In [ ]:
test_csv = PROC_DIR / 'fci_test.csv'
train_csv = PROC_DIR / 'fci_train.csv'
assert test_csv.exists(), f'Missing {test_csv}. Run Notebook 01 first!'

test_df = pd.read_csv(str(test_csv))
train_df = pd.read_csv(str(train_csv))
for d in [test_df, train_df]:
    d['itemReviewed.author.name'] = d['itemReviewed.author.name'].fillna('Unknown')
    d.dropna(subset=['claimReviewed','label_id'], inplace=True)
    d['label_id'] = d['label_id'].astype(int)

logging.info(f'Test: {fmt(len(test_df))}, Train: {fmt(len(train_df))}')

# Encoder for embeddings
encoder = FrozenEncoder().to(device)

def enc_batch(texts, desc, bs=64):
    embs = []
    for i in tqdm(range(0,len(texts),bs), desc=desc):
        embs.append(encoder.encode(texts[i:i+bs], device).cpu())
    return torch.cat(embs)

test_claims = test_df['claimReviewed'].tolist()
test_speakers = test_df['itemReviewed.author.name'].tolist()
test_dates = test_df['datePublished'].astype(str).tolist()
test_labels = test_df['label_id'].values

test_ce = enc_batch(test_claims, 'Test claims')
test_se = enc_batch(test_speakers, 'Test speakers')
test_de = enc_batch(test_dates, 'Test dates')

# Speaker map for SFR
all_sp = list(set(test_speakers + train_df['itemReviewed.author.name'].tolist()))
if len(all_sp) > 5000: all_sp = random.sample(all_sp, 5000)
sp_map = {}
for i in range(0,len(all_sp),64):
    b = all_sp[i:i+64]
    e = encoder.encode(b, device).cpu()
    for s, emb in zip(b,e): sp_map[s] = emb

# Train embeddings for baselines
train_claims_list = train_df['claimReviewed'].tolist()
train_labels_arr = train_df['label_id'].values
train_ce = enc_batch(train_claims_list, 'Train claims')

del encoder; torch.cuda.empty_cache() if torch.cuda.is_available() else None
logging.info('Embeddings ready, encoder freed')

## 4. Load Trained Model

In [ ]:
ckpts = sorted(CKPT_DIR.glob('cl_sdrg_epoch_*.pt'))
assert ckpts, f'No checkpoints in {CKPT_DIR}. Run Notebook 03 first!'
ckpt = torch.load(str(ckpts[-1]), map_location=device, weights_only=False)
logging.info(f'Loaded checkpoint: {ckpts[-1].name} (epoch {ckpt["epoch"]})')

fga = FeatureGatingAgent().to(device)
fusion = GatedFusion().to(device)
classifier = VeracityClassifier().to(device)
fga.load_state_dict(ckpt['model_state_dict']['fga'])
classifier.load_state_dict(ckpt['model_state_dict']['classifier'])
fga.eval(); classifier.eval()
print('✅ Model loaded')

## 5. Classification Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

all_preds, all_probs = [], []
with torch.no_grad():
    for i in range(0, len(test_ce), BATCH_SIZE):
        eq = test_ce[i:i+BATCH_SIZE].to(device)
        es = test_se[i:i+BATCH_SIZE].to(device)
        et = test_de[i:i+BATCH_SIZE].to(device)
        aq,as_,at = fga(eq,es,et)
        eg = fusion(eq,es,et,aq,as_,at)
        logits = classifier(eg)
        all_preds.append(logits.argmax(-1).cpu().numpy())
        all_probs.append(F.softmax(logits,-1).cpu().numpy())

preds = np.concatenate(all_preds)
probs = np.concatenate(all_probs)

names = [ID2LABEL[i] for i in range(NUM_CLASSES)]
print(classification_report(test_labels, preds, target_names=names, zero_division=0))

cls_metrics = {
    'accuracy': accuracy_score(test_labels, preds),
    'macro_f1': f1_score(test_labels, preds, average='macro', zero_division=0),
    'macro_precision': precision_score(test_labels, preds, average='macro', zero_division=0),
    'macro_recall': recall_score(test_labels, preds, average='macro', zero_division=0),
}
cm = confusion_matrix(test_labels, preds)
for k,v in cls_metrics.items(): print(f'  {k}: {v:.4f}')

## 6. Retrieval Metrics (FAISS)

In [ ]:
# Compute gated embeddings for retrieval
gated_embs = []
with torch.no_grad():
    for i in range(0, len(test_ce), BATCH_SIZE):
        eq = test_ce[i:i+BATCH_SIZE].to(device)
        es = test_se[i:i+BATCH_SIZE].to(device)
        et = test_de[i:i+BATCH_SIZE].to(device)
        aq,as_,at = fga(eq,es,et)
        gated_embs.append(fusion(eq,es,et,aq,as_,at).cpu().numpy())
gated = np.concatenate(gated_embs).astype(np.float32)

# FAISS index
g_copy = gated.copy(); faiss.normalize_L2(g_copy)
index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(g_copy)

q = gated.copy(); faiss.normalize_L2(q)
max_k = max(K_VALUES)
D, I = index.search(q, max_k+1)

ret_metrics = {}
for k in K_VALUES:
    recalls, mrrs, ndcgs = [], [], []
    for i in range(len(q)):
        rl = test_labels[I[i]]
        mask = I[i] != i
        rl = rl[mask][:k]
        rel = (rl == test_labels[i])
        recalls.append(float(rel.any()))
        cp = np.where(rel)[0]
        mrrs.append(1.0/(cp[0]+1) if len(cp)>0 else 0.0)
        dcg = sum(r/np.log2(p+2) for p,r in enumerate(rel.astype(float)))
        idcg = sum(1.0/np.log2(p+2) for p in range(max(1,rel.sum())))
        ndcgs.append(dcg/idcg if idcg>0 else 0.0)
    ret_metrics[f'recall@{k}'] = float(np.mean(recalls))
    ret_metrics[f'mrr@{k}'] = float(np.mean(mrrs))
    ret_metrics[f'ndcg@{k}'] = float(np.mean(ndcgs))

print('\nRetrieval Metrics:')
for k in K_VALUES:
    print(f'  K={k:<4} Recall={ret_metrics[f"recall@{k}"]:.4f}  MRR={ret_metrics[f"mrr@{k}"]:.4f}  nDCG={ret_metrics[f"ndcg@{k}"]:.4f}')

## 7. Speaker Flip Rate (SFR) Audit

In [ ]:
print(f'SFR Audit: {fmt(len(test_ce))} samples × {SFR_PERTURBS} perturbations')

all_sp_list = list(sp_map.keys())
n = len(test_ce)
flips = np.zeros(n)

with torch.no_grad():
    # Original predictions
    orig_p = []
    for i in range(0,n,64):
        eq=test_ce[i:i+64].to(device); es=test_se[i:i+64].to(device); et=test_de[i:i+64].to(device)
        aq,as_,at=fga(eq,es,et); eg=fusion(eq,es,et,aq,as_,at)
        orig_p.append(classifier(eg).argmax(-1).cpu())
    orig_p = torch.cat(orig_p)

    for pert in tqdm(range(SFR_PERTURBS), desc='SFR perturbations'):
        rsp = random.choices(all_sp_list, k=n)
        cf_se = torch.stack([sp_map[s] for s in rsp])
        cf_p = []
        for i in range(0,n,64):
            eq=test_ce[i:i+64].to(device); es_cf=cf_se[i:i+64].to(device); et=test_de[i:i+64].to(device)
            aq,as_,at=fga(eq,es_cf,et); eg=fusion(eq,es_cf,et,aq,as_,at)
            cf_p.append(classifier(eg).argmax(-1).cpu())
        cf_p = torch.cat(cf_p)
        flips += (orig_p != cf_p).numpy().astype(float)

sfr = float(np.mean(flips / SFR_PERTURBS))
sfr_passed = sfr < SFR_TARGET

if sfr_passed:
    print(f'\n  ✅ SFR = {sfr:.4f} ({sfr*100:.2f}%) — TARGET MET (< {SFR_TARGET*100:.0f}%)')
else:
    print(f'\n  ⚠️  SFR = {sfr:.4f} ({sfr*100:.2f}%) — TARGET NOT MET (≥ {SFR_TARGET*100:.0f}%)')

## 8. Ablation Baselines

In [ ]:
ablation = []

# ── BM25 ──
try:
    from rank_bm25 import BM25Okapi
    train_tok = [c.lower().split() for c in train_claims_list]
    bm25 = BM25Okapi(train_tok)
    bm25_preds = []
    for c in tqdm(test_claims, desc='BM25'):
        scores = bm25.get_scores(c.lower().split())
        bm25_preds.append(train_labels_arr[scores.argmax()])
    bm25_preds = np.array(bm25_preds)
    bm25_m = {'method': 'BM25', 'accuracy': accuracy_score(test_labels, bm25_preds),
              'macro_f1': f1_score(test_labels, bm25_preds, average='macro', zero_division=0),
              'macro_precision': precision_score(test_labels, bm25_preds, average='macro', zero_division=0),
              'macro_recall': recall_score(test_labels, bm25_preds, average='macro', zero_division=0)}
    ablation.append(bm25_m)
    print(f'BM25: Acc={bm25_m["accuracy"]:.4f}, F1={bm25_m["macro_f1"]:.4f}')
except ImportError:
    print('BM25 skipped (rank_bm25 not installed)')

# ── Zero-Shot kNN ──
tr_np = train_ce.numpy().astype(np.float32)
te_np = test_ce.numpy().astype(np.float32)
tr_copy = tr_np.copy(); faiss.normalize_L2(tr_copy)
te_copy = te_np.copy(); faiss.normalize_L2(te_copy)
zs_idx = faiss.IndexFlatIP(EMBEDDING_DIM); zs_idx.add(tr_copy)
_, zs_I = zs_idx.search(te_copy, 1)
zs_preds = train_labels_arr[zs_I[:,0]]
zs_m = {'method': 'Zero-Shot kNN', 'accuracy': accuracy_score(test_labels, zs_preds),
         'macro_f1': f1_score(test_labels, zs_preds, average='macro', zero_division=0),
         'macro_precision': precision_score(test_labels, zs_preds, average='macro', zero_division=0),
         'macro_recall': recall_score(test_labels, zs_preds, average='macro', zero_division=0)}
ablation.append(zs_m)
print(f'Zero-Shot kNN: Acc={zs_m["accuracy"]:.4f}, F1={zs_m["macro_f1"]:.4f}')

## 9. Results Table & Plots

In [ ]:
# ── Benchmark Table ──
print('\n' + '='*90)
print('  BENCHMARK RESULTS')
print('='*90)
print(f'{"Method":<30} {"Acc":>8} {"F1":>8} {"Prec":>8} {"Rec":>8} {"SFR":>8}')
print('-'*90)
print(f'{"CL-SDRG (Ours)":<30} {cls_metrics["accuracy"]:.4f}   {cls_metrics["macro_f1"]:.4f}   '
      f'{cls_metrics["macro_precision"]:.4f}   {cls_metrics["macro_recall"]:.4f}   {sfr:.4f}')
for r in ablation:
    print(f'{r["method"]:<30} {r["accuracy"]:.4f}   {r["macro_f1"]:.4f}   '
          f'{r["macro_precision"]:.4f}   {r["macro_recall"]:.4f}   {"N/A":>6}')
print('-'*90)

print('\n  RETRIEVAL METRICS (CL-SDRG)')
for k in K_VALUES:
    print(f'  K={k:<4} Recall={ret_metrics[f"recall@{k}"]:.4f}  MRR={ret_metrics[f"mrr@{k}"]:.4f}  nDCG={ret_metrics[f"ndcg@{k}"]:.4f}')
print('='*90)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('CL-SDRG Evaluation Results', fontsize=16, fontweight='bold')

# Classification comparison
methods = ['CL-SDRG'] + [r['method'] for r in ablation]
all_res = [cls_metrics] + ablation
mnames = ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall']
mlabels = ['Accuracy', 'Macro-F1', 'Precision', 'Recall']
x = np.arange(len(mnames))
w = 0.8 / len(methods)
colors = plt.cm.Set2(np.linspace(0, 1, len(methods)))
for i, (m, r) in enumerate(zip(methods, all_res)):
    vals = [r.get(mn, 0) for mn in mnames]
    axes[0].bar(x + i*w, vals, w, label=m, color=colors[i])
axes[0].set_xticks(x + w*(len(methods)-1)/2)
axes[0].set_xticklabels(mlabels)
axes[0].set_ylabel('Score'); axes[0].set_title('Classification Metrics')
axes[0].legend(fontsize=8); axes[0].set_ylim(0,1); axes[0].grid(axis='y', alpha=0.3)

# SFR
bars = axes[1].bar(['CL-SDRG'], [sfr*100], color=['#2ecc71'])
axes[1].axhline(SFR_TARGET*100, color='red', ls='--', lw=2, label=f'Target: {SFR_TARGET*100:.0f}%')
axes[1].set_ylabel('SFR (%)'); axes[1].set_title('Shortcut Bias Audit')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
axes[1].text(0, sfr*100+0.3, f'{sfr*100:.2f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(str(FIG_DIR/'evaluation_results.png'), dpi=150, bbox_inches='tight')
plt.show()

# Confusion matrix
fig, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(cm, cmap='Blues')
ax.set_title('CL-SDRG Confusion Matrix', fontsize=14, fontweight='bold')
lbls = [ID2LABEL[i] for i in range(NUM_CLASSES)]
ax.set_xticks(range(len(lbls))); ax.set_yticks(range(len(lbls)))
ax.set_xticklabels(lbls); ax.set_yticklabels(lbls)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j,i,str(cm[i,j]),ha='center',va='center',
                color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout()
plt.savefig(str(FIG_DIR/'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\n{"="*70}')
print(f'  PHASE 4 COMPLETE')
print(f'  Accuracy: {cls_metrics["accuracy"]:.4f} | F1: {cls_metrics["macro_f1"]:.4f} | SFR: {sfr:.4f} ({"PASS" if sfr_passed else "FAIL"})')
print(f'{"="*70}')

## 10. Export Results

Download the results CSV and figures for your paper.

In [ ]:
# Save results
results = pd.DataFrame([
    {'method': 'CL-SDRG', 'sfr': sfr, **cls_metrics, **ret_metrics},
    *ablation
])
results.to_csv('outputs/benchmark_results.csv', index=False)
pd.DataFrame([{'sfr': sfr, 'target': SFR_TARGET, 'passed': sfr_passed, 'perturbations': SFR_PERTURBS}]).to_csv('outputs/sfr_audit.csv', index=False)
print('📄 Results saved to outputs/')

# Download
try:
    from google.colab import files
    files.download('outputs/benchmark_results.csv')
    files.download('outputs/sfr_audit.csv')
    files.download(str(FIG_DIR/'evaluation_results.png'))
    files.download(str(FIG_DIR/'confusion_matrix.png'))
except: pass

print('\n📋 Copy the benchmark table and share it back for Phase 5 (Publication).')